# SwiftSQL V1 (opt2): SQLite 沙箱去脏 + 极速 Auto-Think 多方言 SFT

> **Colab Pro 省积分建议**：右上角切换 **L4 GPU (24GB)**（~2 CU/小时），本 Notebook 约 30–45 分钟跑完（耗 ~1.5 积分）。
> 所有数据与模型隔离保存在 `MyDrive/swift-sql/opt2/`，不会覆盖你之前训练好的初版模型。


In [ ]:
# 1. 检查 GPU 并挂载 Google Drive
import os, torch
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive", force_remount=True)
else:
    print("Google Drive already mounted at /content/drive")

DRIVE_DIR = "/content/drive/MyDrive/swift-sql"
os.makedirs(DRIVE_DIR, exist_ok=True)

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Ready: {name} ({vram:.1f} GB VRAM) | bf16={torch.cuda.is_bf16_supported()}")
else:
    raise RuntimeError("未检测到 GPU！请在 Colab 菜单：代码执行程序 -> 更改运行时类型 -> 选择 L4 GPU。")


In [ ]:
# 2. 拉取最新 GitHub 仓库并安装依赖
import sys

REPO_URL = "https://github.com/2422725977/swift-sql.git"
REPO_DIR = "/content/swift-sql"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

!pip install -q --upgrade unsloth trl datasets sqlglot pyyaml
!pip install -q --no-deps -e {REPO_DIR}
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")


In [ ]:
# 3. 构建 opt2 干净多任务语料（启用 SQLite 沙箱预过滤 + 极速 Auto-Think 60字规划）
from collections import Counter
from pathlib import Path
import yaml

from swiftsql.data_pipeline import (
    build_sft_records,
    load_jsonl,
    load_public_hf_datasets,
    save_jsonl,
)

PROFILE = "3b"  # 可选 "3b" (opt2, bf16高速) 或 "7b" (opt2_7b, 高精度双开赛马)
CFG_FILE = "train_config_7b.yaml" if PROFILE == "7b" else "train_config.yaml"
with open(f"{REPO_DIR}/configs/{CFG_FILE}", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)
print(f"Active Profile={PROFILE} | Model={CFG['model']['base_model']} | 4bit={CFG['model']['load_in_4bit']}")

RUN_TAG = CFG["project"].get("run_tag", "opt2")
RUN_DIR = Path(DRIVE_DIR) / RUN_TAG
DATA_DIR = RUN_DIR / "data"
TRAIN_PATH = DATA_DIR / "train_sft.jsonl"
EVAL_PATH = DATA_DIR / "eval.jsonl"

FORCE_REBUILD = False

if TRAIN_PATH.exists() and EVAL_PATH.exists() and not FORCE_REBUILD:
    train_records = load_jsonl(TRAIN_PATH)
    eval_records = load_jsonl(EVAL_PATH)
    print(f"[{RUN_TAG}] Loaded cached datasets: train={len(train_records)}, eval={len(eval_records)}")
else:
    dcfg = CFG["data"]
    print(f"[{RUN_TAG}] Downloading open HF datasets & running SQLite sandbox pre-filtering...")
    train_raw, eval_raw = load_public_hf_datasets(
        gretel_train_samples=dcfg["gretel_train_samples"],
        sqale_train_samples=dcfg["sqale_train_samples"],
        chinese_train_samples=dcfg["chinese_train_samples"],
        eval_samples=dcfg["eval_samples"],
        seed=CFG["project"]["seed"],
    )
    train_records = build_sft_records(
        train_raw,
        dialect_weights=dcfg["dialect_weights"],
        repair_ratio=dcfg["repair_ratio"],
        refusal_ratio=dcfg["refusal_ratio"],
        max_schema_chars=dcfg["max_schema_chars"],
        verify_executable=dcfg.get("verify_executable", True),
        seed=CFG["project"]["seed"],
    )
    eval_records = build_sft_records(
        eval_raw,
        dialect_weights=dcfg["dialect_weights"],
        repair_ratio=0.10,
        refusal_ratio=0.15,
        max_schema_chars=dcfg["max_schema_chars"],
        verify_executable=dcfg.get("verify_executable", True),
        seed=CFG["project"]["seed"] + 1,
    )
    save_jsonl(train_records, TRAIN_PATH)
    save_jsonl(eval_records, EVAL_PATH)
    print(f"[{RUN_TAG}] Raw train={len(train_raw)} -> Cleaned multi-task train={len(train_records)}, eval={len(eval_records)}")

print("Train task mix:", Counter(r["task_type"] for r in train_records))
print("Train dialect mix:", Counter(r["dialect"] for r in train_records))
print("Train complexity mix:", Counter(r["complexity"] for r in train_records))
print("Train language mix:", Counter(r["language"] for r in train_records))


In [ ]:
# 4. 加载基座模型并挂载 LoRA
from unsloth import FastLanguageModel

BASE_MODEL = CFG["model"]["base_model"]
MAX_SEQ_LENGTH = CFG["model"]["max_seq_length"]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=CFG["model"]["load_in_4bit"],
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CFG["lora"]["r"],
    target_modules=CFG["lora"]["target_modules"],
    lora_alpha=CFG["lora"]["lora_alpha"],
    lora_dropout=CFG["lora"]["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CFG["project"]["seed"],
)
model.print_trainable_parameters()


In [ ]:
# 5. 启动 opt2 V1 SFT 训练（3 epochs + 自动保留验证集最优 checkpoint）
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

def to_text(records):
    return Dataset.from_list([
        {"text": tokenizer.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=False)}
        for r in records
    ])

train_ds = to_text(train_records)
eval_ds = to_text(eval_records[:250])

sft_cfg = CFG["sft"]
OUT_DIR = str(RUN_DIR / "checkpoints" / "v1_sft")
use_bf16 = torch.cuda.is_bf16_supported()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        output_dir=OUT_DIR,
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        packing=sft_cfg.get("packing", False),
        num_train_epochs=sft_cfg["num_train_epochs"],
        per_device_train_batch_size=sft_cfg["per_device_train_batch_size"],
        gradient_accumulation_steps=sft_cfg["gradient_accumulation_steps"],
        learning_rate=sft_cfg["learning_rate"],
        warmup_ratio=sft_cfg["warmup_ratio"],
        lr_scheduler_type=sft_cfg["lr_scheduler_type"],
        logging_steps=sft_cfg["logging_steps"],
        save_steps=sft_cfg["save_steps"],
        eval_strategy="steps",
        eval_steps=sft_cfg["eval_steps"],
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        save_total_limit=3,
        fp16=not use_bf16,
        bf16=use_bf16,
        optim="adamw_8bit",
        seed=CFG["project"]["seed"],
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print("opt2 V1 SFT finished! Runtime (s):", round(trainer_stats.metrics["train_runtime"], 1))

LORA_DIR = str(RUN_DIR / "models" / "v1_sft_lora")
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print("Saved V1 LoRA adapter to:", LORA_DIR)


In [ ]:
# 6. 四合一快速人工验收：简单直出 / 1-JOIN极速直出 / 缺列拒答 / 报错自修复
import time
from swiftsql.prompts import build_generation_messages, build_repair_messages

FastLanguageModel.for_inference(model)

def ask(messages, max_new_tokens=256):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    t0 = time.perf_counter()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    ms = (time.perf_counter() - t0) * 1000
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return gen.strip(), round(ms, 1)

schema = (
    "CREATE TABLE users (id INT, name TEXT, city TEXT);\n"
    "CREATE TABLE orders (id INT, user_id INT, amount NUMERIC, status TEXT);"
)

tests = [
    ("1. 简单单表(plan应为空)", build_generation_messages(schema, "查询所有北京用户的姓名", dialect="postgresql")),
    ("2. 复杂JOIN+HAVING(应含<=60字极简plan)", build_generation_messages(schema, "统计每个城市已完成订单的总金额大于1000的城市，按总金额降序取前3名", dialect="mysql")),
    ("3. 缺列拒答(answerable=false)", build_generation_messages(schema, "查询所有用户的手机号和邮箱", dialect="postgresql")),
    ("4. 报错自修复(repair_sql)", build_repair_messages(schema, "统计每个用户的订单数", "SELECT user_id, COUNT(*) FORM orders GROUP BY user_id", 'syntax error at or near "FORM"', dialect="postgresql")),
]

for title, msgs in tests:
    ans, ms = ask(msgs)
    print(f"=== {title} ({ms} ms, {len(ans)} chars) ===")
    print(ans, "\n")


In [ ]:
# 7. 可选导出 V1 GGUF（如果马上跑 Notebook 02 做稳态 DPO，保持 False 省时间）
EXPORT_V1_GGUF = False

if EXPORT_V1_GGUF:
    GGUF_DIR = str(RUN_DIR / "exports" / "v1_sft_gguf")
    os.makedirs(GGUF_DIR, exist_ok=True)
    for q in CFG["export"]["quantizations"]:
        model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method=q)
    !ls -lh {GGUF_DIR}
else:
    print("Skipped V1 GGUF export (will export best model at end of Notebook 02).")


In [ ]:
# 释放 Colab Pro GPU 配额（先刷新 Google Drive 缓存，防止权重/GGUF 丢数据）
import os, time
from google.colab import drive, runtime

AUTO_UNASSIGN = True  # 如需保留当前会话继续调试，可改为 False

if AUTO_UNASSIGN:
    print("Syncing and flushing Google Drive before releasing GPU runtime...")
    os.sync()
    time.sleep(5)
    drive.flush_and_unmount()
    print("Google Drive flushed. Releasing Colab GPU runtime now to save compute units!")
    runtime.unassign()
else:
    print("AUTO_UNASSIGN=False: 保持当前会话。用完后请手动点击右上角「断开连接并删除运行时」以停止扣费。")
